In [175]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
import torchbnn as bnn
from tqdm.auto import tqdm




# Установка случайного сида для воспроизводимости
torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [176]:
df = pd.read_csv('../data/data.csv')
df

,"Sigma, Mpa",T.K,t.h,Th,C,Cr,Co,Mo,W,Al,...,La,S,Si,Mn,P,Hf,Cu,Ge,Ga,Ni
0,241.316495,1144.2600,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,NaN,99.500
1,241.316495,1144.2600,113.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.75,NaN,NaN,NaN,99.250
2,241.316495,1144.2600,68.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.05,99.450
3,241.316495,1144.2600,40.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.20,99.300
4,241.316495,1144.2600,32.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.50,99.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3441,1061.792578,922.0389,1183.7,NaN,0.02,17.0,28.4,3.4,1.9,1.03,...,NaN,6.0,0.05,0.05,NaN,NaN,0.02,NaN,NaN,33.904
3442,1103.161120,866.4833,25554.7,NaN,0.02,17.0,28.4,3.4,1.9,1.03,...,NaN,6.0,0.05,0.05,NaN,NaN,0.02,NaN,NaN,33.904
3443,241.316495,1199.8170,183.2,NaN,0.18,10.0,15.0,3.0,NaN,5.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.750
3444,241.316495,1199.8170,153.0,NaN,0.18,10.0,15.0,3.0,NaN,5.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.750


In [177]:
mass = '232,03806	12,011	51,9961	58,933194	95,95	183,84	26,9815385	47,867	92,90637	10,81	55,845	88,90584	91,224	180,94788	186,207	101,07	50,9415	140,116	138,90547	32,06	28,085	54,938044	24,305	30,973761998	178,49	107,8682	63,546	208,9804	207,2	192,22	72,63	69,723	58,6934'.replace(',', '.').split()
element = 'Th	C	Cr	Co	Mo	W	Al	Ti	Nb	B	Fe	Y	Zr	Ta	Re	Ru	V	Ce	La	S	Si	Mn	Mg	P	Hf	Ag	Cu	Bi	Pb	Ir	Ge	Ga	Ni'.split()
atom_mass = dict(zip(element, mass))

In [178]:
atom_mass['Ni']

'58.6934'

In [179]:
for elem in df.drop(columns=['Sigma, Mpa', 'T.K', 't.h']).columns.to_list():
    df[elem] = df[elem] / float(atom_mass[elem])
df['sum'] = df[df.drop(columns=['Sigma, Mpa', 'T.K', 't.h']).columns.to_list()].sum(axis=1, skipna=True)
df['PLM'] = df['T.K'] * (20 + np.log10(df['t.h'])) * 1e-5

cols = df.drop(columns=['Sigma, Mpa', 'T.K', 't.h']).columns
df.loc[:, cols] = df.loc[:, cols].div(df['sum'], axis=0)
df.loc[:, cols] = df.loc[:, cols].div(df['Ni'], axis=0)



df = df.fillna(0)

df = df.drop(columns=['T.K', 't.h', 'Ni', 'sum'])

df

,"Sigma, Mpa",Th,C,Cr,Co,Mo,W,Al,Ti,Nb,...,La,S,Si,Mn,P,Hf,Cu,Ge,Ga,PLM
0,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001652,0.000000,0.0,0.000000,0.139405
1,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.002485,0.000000,0.0,0.000000,0.149239
2,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001653,0.000000,0.0,0.000423,0.147465
3,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001656,0.000000,0.0,0.001695,0.146154
4,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001661,0.000000,0.0,0.004252,0.145925
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3441,1061.792578,0.0,0.002883,0.566000,0.834251,0.061344,0.017892,0.066086,0.112115,0.020497,...,0.0,0.323986,0.003082,0.001576,0.0,0.000000,0.000545,0.0,0.000000,0.368295
3442,1103.161120,0.0,0.002883,0.566000,0.834251,0.061344,0.017892,0.066086,0.112115,0.020497,...,0.0,0.323986,0.003082,0.001576,0.0,0.000000,0.000545,0.0,0.000000,0.366118
3443,241.316495,0.0,0.014721,0.188921,0.250025,0.030713,0.000000,0.200238,0.112870,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.262391
3444,241.316495,0.0,0.014721,0.188921,0.250025,0.030713,0.000000,0.200238,0.112870,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.261469


In [180]:

def log_transform_target(sigma):
    return -np.log10(np.maximum(sigma, 1e-8))

def inverse_log_transform(y):
    return 10 ** (-y)

def calculate_mse(y_pred, y_true):
    """Вычисление MSE (Mean Squared Error)"""
    return torch.mean((y_pred - y_true) ** 2).item()

def calculate_rmse_absolute(sigma_pred, sigma_real):
    """
    Вычисление RMSE для физических значений σ (MPa)
    Корень из среднего квадрата абсолютных ошибок
    """
    mse = torch.mean((sigma_pred - sigma_real) ** 2).item()
    return np.sqrt(mse)

def calculate_rmse_relative(sigma_pred, sigma_real):
    """
    Вычисление RMSE согласно формуле (6) из статьи.
    Это относительная ошибка (Root Mean Square Relative Error)
    """
    epsilon = 1e-8
    relative_error = (sigma_pred - sigma_real) / (sigma_real.abs() + epsilon)
    mse_relative = torch.mean(relative_error ** 2).item()
    return np.sqrt(mse_relative)

def calculate_metrics_np(pred, real):
    eps = 1e-8
    # Абсолютная RMSE
    rmse_abs = np.sqrt(np.mean((pred - real) ** 2))
    
    # Относительная RMSE (Формула 6 из статьи)
    rel_err = (pred - real) / (real + eps)
    rmse_rel = np.sqrt(np.mean(rel_err ** 2))
    
    # MAPE
    mape = np.mean(np.abs(rel_err)) * 100
    
    # R²
    ss_res = np.sum((real - pred) ** 2)
    ss_tot = np.sum((real - np.mean(real)) ** 2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    
    return rmse_abs, rmse_rel, mape, r2

In [181]:
target_col = 'Sigma, Mpa'
sigma = np.asarray(df[target_col], dtype=np.float32).reshape(-1, 1)
y = np.asarray(log_transform_target(sigma), dtype=np.float32).reshape(-1, 1).astype("float32")
X = df.copy().drop(columns=[target_col]).to_numpy().astype("float32")
X, y

(array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.13940506],
        [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.1492392 ],
        [0.        , 0.        , 0.        , ..., 0.        , 0.00042323,
         0.14746492],
        ...,
        [0.        , 0.01472125, 0.18892115, ..., 0.        , 0.        ,
         0.2623908 ],
        [0.        , 0.01472125, 0.18892115, ..., 0.        , 0.        ,
         0.26146874],
        [0.        , 0.01472125, 0.18892115, ..., 0.        , 0.        ,
         0.2588929 ]], shape=(3446, 27), dtype=float32),
 array([[-2.3825872],
        [-2.3825872],
        [-2.3825872],
        ...,
        [-2.3825872],
        [-2.3825872],
        [-2.4405792]], shape=(3446, 1), dtype=float32))

In [182]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    shuffle=True
)


In [ ]:
class SimpleANN(nn.Module):
    def __init__(self, input_size=24, hidden_size=13):
        super(SimpleANN, self).__init__()
        self.hidden = nn.Linear(input_size, hidden_size)
        self.activation = nn.Tanh()
        self.output = nn.Linear(hidden_size, 1)
        
        # Инициализация весов
        nn.init.normal_(self.hidden.weight, mean=0, std=0.01)
        nn.init.zeros_(self.hidden.bias)
        nn.init.normal_(self.output.weight, mean=0, std=0.01)
        nn.init.zeros_(self.output.bias)

    def forward(self, x):
        return self.output(self.activation(self.hidden(x)))

In [184]:
import copy


def train_model(model, X_train, y_train, X_val, y_val, 
                weight_decay=0.0, epochs=200, lr=0.01, verbose=False):
    """
    Обучает модель с заданным коэффициентом регуляризации (weight_decay)
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    X_tr = torch.FloatTensor(X_train).to(device)
    y_tr = torch.FloatTensor(y_train).view(-1, 1).to(device)
    X_vl = torch.FloatTensor(X_val).to(device)
    y_vl = torch.FloatTensor(y_val).view(-1, 1).to(device)
    
    # Weight decay — это и есть наша регуляризация (аналог alpha/lambda)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()
    
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        
        outputs = model(X_tr)
        loss = criterion(outputs, y_tr)
        loss.backward()
        optimizer.step()
        
        # Оценка на валидации
        model.eval()
        with torch.no_grad():
            val_pred = model(X_vl)
            val_loss = criterion(val_pred, y_vl).item()
        
        # Сохранение лучшей модели
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Early stopping (если валидация не улучшается 20 эпох)
        if patience_counter >= 20:
            if verbose: print(f"  Early stopping at epoch {epoch}")
            break

    # Восстановление лучших весов
    if best_model_state:
        model.load_state_dict(best_model_state)
    
    return model, best_val_loss

## main

In [185]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X_temp).astype("float32") # ???
y_scaled = scaler_y.fit_transform(y_temp).astype("float32").reshape(-1,1)

X_test_scaled = scaler_X.transform(X_test).astype("float32")
y_test_scaled = scaler_y.transform(y_test).astype("float32").reshape(-1,1)

X_train, X_val, y_train, y_val = train_test_split(X_scaled, y_scaled, test_size=0.25, random_state=42)




In [ ]:
lambdas = [0.0, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]

results = {}
best_lambda = None
min_val_loss = float('inf')
best_model = None

print(f"\n{'Lambda (WD)':>12} | {'Val Loss':>10} | {'Status'}")
print("-" * 35)

# 4. ЦИКЛ ПОДБОРА ГИПЕРПАРАМЕТРА
for lam in lambdas:
    model = SimpleANN(input_size=27, hidden_size=10)
    
    # Обучаем модель с текущим lambda
    trained_model, val_loss = train_model(
        model, X_train, y_train, X_val, y_val,
        weight_decay=lam, epochs=300, lr=0.05
    )
    
    results[lam] = val_loss
    status = "✅ BEST" if val_loss < min_val_loss else ""
    print(f"{lam:>12.6f} | {val_loss:>10.6f} | {status}")
    
    if val_loss < min_val_loss:
        min_val_loss = val_loss
        best_lambda = lam
        best_model = trained_model

print(f"\n🏆 Лучший коэффициент регуляризации (weight_decay): {best_lambda}")


 Lambda (WD) |   Val Loss | Status
-----------------------------------
    0.000000 |   0.301528 | ✅ BEST
    0.000010 |   0.301820 | 
    0.000100 |   0.327450 | 
    0.001000 |   0.307534 | 
    0.010000 |   0.322697 | 
    0.100000 |   0.371352 | 
    1.000000 |   0.999884 | 

🏆 Лучший коэффициент регуляризации (weight_decay): 0.0


In [187]:
print("\n" + "="*60)
print("ФИНАЛЬНАЯ ОЦЕНКА НА ТЕСТОВЫХ ДАННЫХ")
print("="*60)

best_model.eval()
with torch.no_grad():
    X_test_t = torch.FloatTensor(X_test_scaled)
    y_pred_scaled = best_model(X_test_t).numpy()

# Обратная трансформация
y_test_real = y_test # y_test уже в логарифмах, т.к. scaler применяли только к X
y_pred = scaler_y.inverse_transform(y_pred_scaled)
sigma_pred = inverse_log_transform(y_pred)
sigma_real_final = inverse_log_transform(y_test_real)

# Метрики
rmse = np.sqrt(np.mean((sigma_pred - sigma_real_final)**2))
mape = np.mean(np.abs((sigma_pred - sigma_real_final) / (sigma_real_final + 1e-8))) * 100

# R2
ss_res = np.sum((sigma_real_final - sigma_pred) ** 2)
ss_tot = np.sum((sigma_real_final - np.mean(sigma_real_final)) ** 2)
r2 = 1 - (ss_res / ss_tot)

print(f"RMSE: {rmse:.2f} MPa")
print(f"MAPE: {mape:.2f} %")
print(f"R2:   {r2:.4f}")
print(f"Pred Range: [{sigma_pred.min():.1f}, {sigma_pred.max():.1f}] MPa")
print(f"Real Range: [{sigma_real_final.min():.1f}, {sigma_real_final.max():.1f}] MPa")


ФИНАЛЬНАЯ ОЦЕНКА НА ТЕСТОВЫХ ДАННЫХ
RMSE: 132.17 MPa
MAPE: 35.00 %
R2:   0.6669
Pred Range: [30.2, 1291.1] MPa
Real Range: [10.0, 1279.0] MPa
